# XGBoost y LightGBM
## Forecasting como problema de regresión

> **Fase 3 — Video 15** | Modelado y Pronóstico
> Dataset: Histórico de Ventas Sintético

---

### El cambio de paradigma

Hasta ahora modelamos *una serie a la vez* (ARIMA, Prophet). El gradient boosting cambia el chip:
**convertimos la serie en un dataset tabular** —una fila por SKU-semana con sus features— y entrenamos
**un solo modelo global** para todos los SKUs a la vez. Es el enfoque del **tier B** del Video 7: en vez de
500 modelos, uno que aprende patrones compartidos.

Aquí se paga toda la Fase 2: calendario (Video 8), lags/ventanas (Video 9) y precio/promoción (Video 10)
son, literalmente, las columnas del modelo. Y responderemos, con honestidad: **¿le gana al SARIMAX?**

### Ruta del notebook
1. Librerías y carga
2. De serie a tabla: la matriz supervisada
3. El problema del multi-step (recursivo vs directo)
4. XGBoost global: un modelo para todos los SKUs
5. ¿Qué mira el modelo? Importancia de features
6. LightGBM: el hermano rápido
7. Veredicto vs. SARIMAX y el benchmark
8. Conclusiones y puente al Video 16

---
## 1. Librerías y carga de datos

In [ ]:
import warnings, sys
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from pathlib import Path

plt.rcParams.update({
    'figure.facecolor': '#F8FAFC', 'axes.facecolor': '#F8FAFC',
    'axes.grid': True, 'grid.color': '#E2E8F0',
    'grid.linewidth': 0.8, 'font.size': 11,
})
BLUE, RED, GREEN, PURPLE, ORANGE = '#2563EB','#DC2626','#16A34A','#7C3AED','#EA580C'
unit_fmt = FuncFormatter(lambda x, _: f'{x/1e3:.0f}k')
print('✅ Librerías cargadas')

In [ ]:
def find_csv(filename='sales_history.csv', max_levels=4):
    base = Path().resolve()
    for _ in range(max_levels):
        candidate = base / 'output' / filename
        if candidate.exists():
            return candidate
        base = base.parent
    raise FileNotFoundError(f"No se encontró '{filename}'. Corre main.py primero.")

csv_path = find_csv()
sys.path.insert(0, str(csv_path.parents[1]))
from src.features import make_supervised
from src.metrics import wape, mase

df = pd.read_csv(csv_path, parse_dates=['date']).sort_values('date')
catalog = pd.read_csv(find_csv('sku_catalog.csv')).set_index('sku_id')

sup = make_supervised(df, catalog, freq='W-MON', lags=(1, 4, 52), windows=(4, 12))
# quitamos la primera semana parcial
sup = sup[sup['date'] > sup['date'].min()]
print(f'✅ Matriz supervisada: {len(sup):,} filas (SKU × semana) × {sup.shape[1]} columnas')

---
## 2. De serie a tabla: la matriz supervisada

El truco es reformular el forecasting como **regresión tabular**: la variable objetivo `y` (demanda) y, en
cada fila, todo lo que el modelo puede saber para predecirla — calendario, precio/promo, memoria (lags) y
atributos del SKU. La función `src.features.make_supervised` lo arma reutilizando el calendario del
generador (una sola fuente de verdad).

In [ ]:
cols_show = ['sku_id', 'date', 'y', 'promo', 'log_price', 'month', 'cal_event',
             'cal_quincena', 'lag_52', 'rollmean_4', 'category', 'base_demand']
print('Una muestra de la matriz (un SKU regular):')
sku0 = df.groupby('sku_id')['revenue'].sum().idxmax()
print(sup[sup['sku_id'] == sku0][cols_show].dropna().head(6).to_string(index=False))
print(f'\nCada fila es un (SKU, semana). Un solo modelo aprende de las {sup["sku_id"].nunique()} series a la vez.')

---
## 3. El problema del multi-step

Aquí está el matiz que hunde a los principiantes. Para pronosticar **52 semanas hacia adelante** no puedes
usar `lag_1` (la demanda de la semana pasada): en la semana +10 del futuro, *no existe todavía*. Tres
estrategias:

| Estrategia | Idea | Costo |
|---|---|---|
| **Recursiva** | Predices +1, la usas como `lag_1` para +2, y así | Acumula error |
| **Directa** | Un modelo por horizonte, con features conocidas | Muchos modelos |
| **Features horizonte-seguras** | Usa solo lo conocible a futuro (`lag_52`, calendario, promo) | Simple y robusto |

Usamos la tercera: features que **sí** están disponibles al pronosticar. Para un horizonte de 52 semanas,
`lag_52` (la demanda de hace un año) ya es histórico conocido, y el calendario/promo se planean por
adelantado. Sin recursión, sin fuga.

In [ ]:
sup = pd.get_dummies(sup, columns=['category'])
cat_cols = [c for c in sup.columns if c.startswith('category_')]

# Features horizonte-seguras (conocidas al pronosticar a 52 semanas)
FEATURES = (['month', 'weekofyear', 'cal_event', 'cal_quincena', 'cal_holiday',
             'promo', 'log_price', 'base_demand', 'lag_52'] + cat_cols)

dates = np.sort(sup['date'].unique())
split = dates[-52]                       # último año como holdout
train = sup[sup['date'] < split].dropna(subset=FEATURES + ['y'])
test  = sup[sup['date'] >= split].dropna(subset=FEATURES + ['y'])
print(f'Train: {len(train):,} filas | Test: {len(test):,} filas ({52} semanas × SKUs)')
print(f'Features ({len(FEATURES)}): {", ".join(FEATURES[:9])} …')

---
## 4. XGBoost global: un modelo para todos los SKUs

Entrenamos **un** XGBoost sobre las filas de todos los SKUs y pronosticamos el holdout de golpe (directo).
Luego agregamos las predicciones por SKU al total semanal para compararlo con los modelos anteriores.

In [ ]:
xgb = XGBRegressor(n_estimators=400, max_depth=6, learning_rate=0.05,
                   subsample=0.8, colsample_bytree=0.8, random_state=0, n_jobs=-1)
xgb.fit(train[FEATURES], train['y'])
test = test.copy()
test['pred_xgb'] = xgb.predict(test[FEATURES]).clip(min=0)

# Agregado al total semanal
tot = test.groupby('date').agg(real=('y', 'sum'), xgb=('pred_xgb', 'sum'))
tot_train = train.groupby('date')['y'].sum()
snaive_tot = pd.Series(tot_train.iloc[-52:].values[:len(tot)], index=tot.index)

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(tot.index, tot['real'], color='black', linewidth=2.5, label='REAL')
ax.plot(tot.index, tot['xgb'], color=PURPLE, linewidth=1.8, label='XGBoost global')
ax.plot(tot.index, snaive_tot.values, color=GREEN, linewidth=1.2, alpha=0.7, label='Seasonal Naive')
ax.yaxis.set_major_formatter(unit_fmt)
ax.set_title('XGBoost global: un modelo, todos los SKUs, agregado al total', fontsize=13, fontweight='bold')
ax.legend(loc='upper left')
plt.tight_layout()
plt.show()

print(f'XGBoost global (total): WAPE {wape(tot.real, tot.xgb):.3f} | MASE {mase(tot.real, tot.xgb, tot_train, 52):.2f}')

# También sirve por SKU: dos ejemplos desde el MISMO modelo
fig, axes = plt.subplots(1, 2, figsize=(15, 4))
for ax, s in zip(axes, [sku0, test['sku_id'].unique()[3]]):
    d = test[test['sku_id'] == s]
    ax.plot(d['date'], d['y'], color='black', linewidth=2, label='real')
    ax.plot(d['date'], d['pred_xgb'], color=PURPLE, linewidth=1.6, label='XGBoost')
    ax.set_title(s, fontsize=11, fontweight='bold'); ax.legend()
fig.suptitle('El mismo modelo global pronostica cada SKU', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 5. ¿Qué mira el modelo? Importancia de features

Una ventaja del boosting: nos dice qué features usa. Debería premiar lo que sabemos que importa —
`lag_52` (memoria anual), el calendario y la promoción.

In [ ]:
imp = pd.Series(xgb.feature_importances_, index=FEATURES).sort_values()
imp = imp[imp > 0.005]

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(imp.index, imp.values, color=BLUE)
ax.set_title('Importancia de features (XGBoost global)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('Top features:')
for f, v in imp.sort_values(ascending=False).head(6).items():
    print(f'  {f:<16}: {v:.3f}')
print('\nComo esperábamos: la memoria anual (lag_52), el nivel del SKU (base_demand) y el')
print('calendario/promo dominan — exactamente los features que construimos en la Fase 2.')

---
## 6. LightGBM: el hermano rápido

LightGBM es otro gradient boosting, típicamente **más rápido** en datasets grandes (crece los árboles por
hojas). La API es casi idéntica; el resultado, muy parecido.

In [ ]:
lgbm = LGBMRegressor(n_estimators=400, max_depth=6, learning_rate=0.05,
                     subsample=0.8, colsample_bytree=0.8, random_state=0, n_jobs=-1, verbose=-1)
lgbm.fit(train[FEATURES], train['y'])
test['pred_lgbm'] = lgbm.predict(test[FEATURES]).clip(min=0)
tot['lgbm'] = test.groupby('date')['pred_lgbm'].sum()

print(f'XGBoost  (total): WAPE {wape(tot.real, tot.xgb):.3f} | MASE {mase(tot.real, tot.xgb, tot_train, 52):.2f}')
print(f'LightGBM (total): WAPE {wape(tot.real, tot.lgbm):.3f} | MASE {mase(tot.real, tot.lgbm, tot_train, 52):.2f}')
print('Prácticamente empatados: la elección entre ambos suele ser por velocidad y ecosistema.')

---
## 7. Veredicto vs. SARIMAX y el benchmark

La pregunta honesta de siempre.

In [ ]:
final = pd.DataFrame([
    {'modelo': 'Seasonal Naive', 'WAPE': wape(tot.real, snaive_tot), 'MASE': mase(tot.real, snaive_tot, tot_train, 52)},
    {'modelo': 'SARIMAX (V12)',  'WAPE': 0.049, 'MASE': 0.56},
    {'modelo': 'XGBoost global', 'WAPE': wape(tot.real, tot.xgb),   'MASE': mase(tot.real, tot.xgb, tot_train, 52)},
    {'modelo': 'LightGBM global','WAPE': wape(tot.real, tot.lgbm),  'MASE': mase(tot.real, tot.lgbm, tot_train, 52)},
]).set_index('modelo')

fig, ax = plt.subplots(figsize=(10, 4.5))
colors = [GREEN if v < 1 else RED for v in final['MASE']]
ax.barh(final.index[::-1], final['MASE'][::-1], color=colors[::-1])
ax.axvline(1.0, color='black', linestyle='--', linewidth=1.5, label='listón (seasonal naive)')
ax.set_title('MASE por modelo (< 1 le gana al benchmark)', fontsize=13, fontweight='bold')
ax.legend()
for i, v in enumerate(final['MASE'][::-1]):
    ax.text(v, i, f' {v:.2f}', va='center', fontweight='bold')
plt.tight_layout()
plt.show()

print(final.assign(WAPE=lambda d: (d.WAPE * 100).round(1), MASE=lambda d: d.MASE.round(2)).to_string())
print('\nHonestidad: XGBoost le gana al benchmark (MASE < 1) pero NO al SARIMAX afinado en esta serie')
print('agregada. Su ventaja NO es la precisión aquí, sino la ESCALA: un solo modelo para las 50 series,')
print('que además captura no-linealidades e interacciones y absorbe features nuevos sin re-especificar.')

---
## 8. Conclusiones y puente al Video 16

### Las reglas que te llevas

1. **Forecasting como regresión tabular:** una fila por SKU-período, y la Fase 2 se vuelve tus columnas.
2. **Un modelo global > muchos locales** cuando tienes cientos de SKUs (tier B). XGBoost aprende patrones
   compartidos y pronostica cualquier SKU desde un solo modelo.
3. **Cuidado con el multi-step:** no puedes usar `lag_1` a 52 semanas. Usa features horizonte-seguras
   (`lag_52`, calendario, promo) o esquemas recursivo/directo — sin fuga.
4. **XGBoost ≈ LightGBM;** elige por velocidad/ecosistema.
5. **ML no es automáticamente mejor:** aquí no superó al SARIMAX afinado. Su fuerza es la escala, las
   no-linealidades y la facilidad de sumar features — no un MASE mágico. Mídelo, como siempre.

### El problema que ninguno resolvió aún

Hemos pronosticado SKUs por separado (y el total por agregación). Pero en Supply Chain necesitas que los
pronósticos de SKU, categoría y total **cuadren entre sí** — y casi nunca lo hacen.

---

### Próximo video
**Video 16 — Forecasting jerárquico: bottom-up, top-down y reconciliación MinT**
Cómo hacer que 50 SKUs, 5 categorías y el total sean coherentes — con reconciliación óptima.